# CLIP ViT-B/16 image-only model - 5-fold cross-validation (Kaggle)

**Before running**: Accelerator **GPU**, Internet **On**. Code and processed data are cloned from GitHub (push your latest changes first).

The model sees **only the thumbnail**. It is trained on the same 5 folds as the text models, so its out-of-fold score is directly comparable with the text baseline (out-of-fold F1 0.771).

| run | what is trained |
|---|---|
| `frozen` | CLIP encoder fixed, only a linear head (linear probe) |
| `finetune_top4` | the head plus the top 4 transformer layers of CLIP |

Reference points for F1 on the clickbait class: predicting clickbait for every article gives about 0.48; a model with no signal that uses the weighted loss lands around there or lower.

In [ ]:
!pip install -q pyvi

In [ ]:
import sys
from pathlib import Path

REPO_URL = "https://github.com/Zaidve/TMG_project_re.git"
CLONE_DIR = Path("/tmp/TMG_project_re")  # outside /kaggle/working so it is not saved as output
ON_KAGGLE = Path("/kaggle/input").exists()
SEARCH_ROOTS = [Path("/kaggle/input"), CLONE_DIR]

def find(pattern):
    return [hit for root in SEARCH_ROOTS if root.exists() for hit in sorted(root.rglob(pattern))]

def find_parent(pattern, levels):
    """Directory `levels` above the first matching file in an attached dataset or the clone."""
    hits = find(pattern)
    if not hits:
        raise FileNotFoundError(f"{pattern} not found under {[str(r) for r in SEARCH_ROOTS]}")
    return hits[0].parents[levels]

if ON_KAGGLE and not find("trainer/train_image_cv.py"):
    !git clone --depth 1 {REPO_URL} {CLONE_DIR}

if ON_KAGGLE:
    CODE_DIR = find_parent("trainer/train_image_cv.py", 1)
    DATA_DIR = find_parent("train.jsonl", 0)
    OUT_DIR = Path("/kaggle/working/clip_cv")
else:  # local run from the notebooks/ folder
    CODE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DATA_DIR = CODE_DIR / "data" / "processed"
    OUT_DIR = CODE_DIR / "outputs" / "clip_cv"

sys.path.insert(0, str(CODE_DIR))
print("code:", CODE_DIR)
print("data:", DATA_DIR)
print("out: ", OUT_DIR)

In [ ]:
import torch

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## Train

Roughly 20-25 minutes per run on a T4 (image decoding is the slow part). Remove an entry from `RUNS` to skip it.

In [ ]:
from trainer.train_image_cv import ImageConfig, run

BASE = dict(
    data_dir=str(DATA_DIR),
    n_folds=5,
    image_size=224,
    augment=True,
    batch_size=64,
    head_lr=1e-3,
    weighted_loss=True,
    patience=3,
    monitor="loss",
    seed=42,
    num_workers=4 if ON_KAGGLE else 0,
    keep_checkpoints=False,
    limit=0,              # set to e.g. 64 for a quick smoke test
)
RUNS = {
    "frozen": dict(unfreeze_layers=0, epochs=10),
    "finetune_top4": dict(unfreeze_layers=4, lr=1e-5, layer_decay=0.9, epochs=6),
}

results = {}
for name, overrides in RUNS.items():
    print(f"################ {name} ################")
    results[name] = run(ImageConfig(**BASE, **overrides, out_dir=str(OUT_DIR / name)))

## Compare with the text baseline

Judge by `oof_f1` (3,067 articles). `always_clickbait` is the score of flagging everything.

In [ ]:
import json
import numpy as np
import pandas as pd
from trainer.trainer import classification_metrics

TEXT_DIR = CODE_DIR / "outputs" / "text_baseline"      # PhoBERT, same folds
text = None
if (TEXT_DIR / "oof_predictions.csv").exists():
    text = {"oof": pd.read_csv(TEXT_DIR / "oof_predictions.csv"), "test": pd.read_csv(TEXT_DIR / "test_predictions.csv"),
            "summary": json.loads((TEXT_DIR / "summary.json").read_text())}

def scores(table, probs=None, threshold=0.5):
    probs = table["prob_clickbait"] if probs is None else probs
    return classification_metrics((table["label"] == "clickbait").to_numpy(), (probs >= threshold).to_numpy())

first = next(iter(results.values()))
rows = {"always_clickbait": {"oof_f1": scores(first["oof_predictions"], pd.Series(1.0, index=first["oof_predictions"].index))["f1"]}}
if text:
    s = text["summary"]
    rows["text (PhoBERT)"] = {"oof_f1": s["oof"]["f1"], "oof_precision": s["oof"]["precision"], "oof_recall": s["oof"]["recall"],
                              "single_test_f1": s["single_model_test_mean"]["f1"], "ensemble_test_f1": s["ensemble_test"]["f1"]}
for name, r in results.items():
    s = r["summary"]
    rows[f"image: {name}"] = {"oof_f1": s["oof"]["f1"], "oof_precision": s["oof"]["precision"], "oof_recall": s["oof"]["recall"],
                              "single_test_f1": s["single_model_test_mean"]["f1"], "ensemble_test_f1": s["ensemble_test"]["f1"]}
comparison = pd.DataFrame(rows).T
display(comparison.round(4))

SHOW = max(results, key=lambda n: results[n]["summary"]["oof"]["f1"])   # run detailed below; set a name to override
result = results[SHOW]
print("showing details for:", SHOW)

## Does the image add anything to the text?

A preview of fusion: the image model's probabilities are averaged with the text model's, with the weight chosen on the out-of-fold predictions. If the best weight is 1.0 (text only) or the gain is under about a point, the thumbnails add little that the text does not already have.

In [ ]:
if text is None:
    print("outputs/text_baseline not found in the repo - skipping")
else:
    def paired(a, b):
        return a.merge(b[["id", "prob_clickbait"]], on="id", suffixes=("_text", "_image"), validate="one_to_one")

    oof_pair = paired(text["oof"], result["oof_predictions"])
    test_pair = paired(text["test"], result["test_predictions"])
    mix = lambda t, w: w * t["prob_clickbait_text"] + (1 - w) * t["prob_clickbait_image"]
    grid = np.arange(0, 1.01, 0.1).round(1)
    weights = pd.DataFrame({"oof_f1": [scores(oof_pair, mix(oof_pair, w))["f1"] for w in grid],
                            "test_f1": [scores(test_pair, mix(test_pair, w))["f1"] for w in grid]},
                           index=pd.Index(grid, name="text_weight"))
    W = weights["oof_f1"].idxmax()
    display(weights.round(4).T)
    print(f"best text weight on out-of-fold: {W}")
    print(f"out-of-fold F1: text {weights.loc[1.0, 'oof_f1']:.4f} | image {weights.loc[0.0, 'oof_f1']:.4f} | mixed {weights.loc[W, 'oof_f1']:.4f}")
    print(f"test F1       : text {weights.loc[1.0, 'test_f1']:.4f} | image {weights.loc[0.0, 'test_f1']:.4f} | mixed {weights.loc[W, 'test_f1']:.4f}")

    t_ok = (oof_pair["prob_clickbait_text"] >= 0.5) == (oof_pair["label"] == "clickbait")
    i_ok = (oof_pair["prob_clickbait_image"] >= 0.5) == (oof_pair["label"] == "clickbait")
    print("\nout-of-fold articles by which model is right:")
    display(pd.crosstab(t_ok.map({True: "text right", False: "text wrong"}), i_ok.map({True: "image right", False: "image wrong"})))
    print("correlation of the two probabilities:", round(oof_pair["prob_clickbait_text"].corr(oof_pair["prob_clickbait_image"]), 3))

## Per-fold results

`val_*` is each model on its own held-out fold; `test_*` is the same model alone on the test split.

In [ ]:
folds = result["fold_metrics"]
display(folds.round(4))
folds.agg(["mean", "std"]).round(4)

## Cross-validated score vs single model vs ensemble

In [ ]:
summary = pd.DataFrame(result["summary"]).T.loc[["oof", "single_model_test_mean", "single_model_test_std", "ensemble_test"]]
summary.index = ["out-of-fold (pooled train+val)", "single model on test (mean)", "single model on test (std)", "ensemble on test"]
summary.round(4)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar([f"fold {i}" for i in folds.index], folds["test_f1"], color="#9ecae1", label="single model")
ax.axhline(result["summary"]["ensemble_test"]["f1"], color="#E45756", linestyle="--", label="ensemble")
ax.set_ylim(max(0, folds["test_f1"].min() - 0.1), 1)
ax.set_ylabel("test F1 (clickbait)")
ax.legend(frameon=False)
fig.tight_layout()

## Ensemble on test

In [ ]:
test = result["test_predictions"]
pd.crosstab(test["label"], test["pred"], rownames=["actual"], colnames=["predicted"], margins=True)

### Threshold

Chosen on the out-of-fold predictions (never on test), then applied to the ensemble.

In [ ]:
oof = result["oof_predictions"]
oof_y, test_y = (oof["label"] == "clickbait").to_numpy(), (test["label"] == "clickbait").to_numpy()

rows = []
for t in np.arange(0.2, 0.81, 0.05):
    o = classification_metrics(oof_y, (oof["prob_clickbait"] >= t).to_numpy())
    e = classification_metrics(test_y, (test["prob_clickbait"] >= t).to_numpy())
    rows.append({"threshold": round(t, 2), "oof_precision": o["precision"], "oof_recall": o["recall"], "oof_f1": o["f1"],
                 "test_precision": e["precision"], "test_recall": e["recall"], "test_f1": e["f1"]})
thresholds = pd.DataFrame(rows).set_index("threshold")
print("best threshold on out-of-fold F1:", thresholds["oof_f1"].idxmax())
thresholds.round(3)

### Out-of-fold results per source and category

Computed on all pooled articles, so the groups are large enough to compare.

In [ ]:
def by_group(table, col):
    rows = {}
    for name, g in table.groupby(col):
        m = classification_metrics((g["label"] == "clickbait").to_numpy(), (g["pred"] == "clickbait").to_numpy())
        rows[name] = {"n": len(g), "clickbait_rate": (g["label"] == "clickbait").mean(), **m}
    return pd.DataFrame(rows).T.round(3).sort_values("n", ascending=False)

by_group(oof, "source")

In [ ]:
by_group(oof, "category")

### Most confident out-of-fold mistakes

Each prediction comes from a model that never saw the article, so confident disagreements are good candidates for a label review.

In [ ]:
pd.set_option("display.max_colwidth", 140)
wrong = oof[oof["label"] != oof["pred"]].copy()
wrong["confidence"] = (wrong["prob_clickbait"] - 0.5).abs() + 0.5
print(len(wrong), "out-of-fold mistakes;", (wrong["confidence"] >= 0.9).sum(), "with confidence >= 0.9")
wrong.sort_values("confidence", ascending=False).head(30)[["id", "source", "title", "label", "pred", "prob_clickbait"]]

## Saved files

`summary.json`, `fold_metrics.csv`, `oof_predictions.csv`, `test_predictions.csv`, per-fold `history.json`. On Kaggle they are under `/kaggle/working/clip_cv/<run name>` and appear in the Output tab after **Save Version**.

In [ ]:
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        print(f"{p.stat().st_size / 1e6:8.1f} MB  {p.relative_to(OUT_DIR)}")